In [31]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
# os.environ["GOOGLE_CSE_ID"] = os.getenv("GOOGLE_CSE_ID")
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")

### __Semantic Search__

In [32]:
from langchain_core.documents import Document

# Document has three attributes [page_content, metadata, id]
# Document object often represents a chunk of a larger document.

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [33]:
# Get full file name
from pathlib import Path

file_name = "KapilDev_C++Dev_8_Resume_Updated.pdf"
# file_name = "Eb_Notice.pdf"
file_path = Path.cwd() / file_name

print(file_path)


c:\Project\LangChain_Documentation\KapilDev_C++Dev_8_Resume_Updated.pdf


In [34]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path)
pdfdocs = loader.load()

In [35]:
# Printing document attributes [metadata, content, id]
print(f"Document length: {len(pdfdocs)}")

# for index, doc in enumerate(pdfdocs):
#     print(f"{index}. Document metadata: {doc.metadata['source']}")
#     print(f"{index}. Document content: {doc.page_content}") 
#     print(f"{index}. Document id: {doc.id}")
#     print("-"*100, "\n")

Document length: 4


In [36]:
# Split the documents
#   - LLM Context window limitation
#   - Effective vector embedding
#       - Improved relevance [More precise results. Help in query phase]
#       - Reduced noise [Combining too much information will dilute the real meaning]
#   - Optima RAG
#       - Splitting documents will help us to retrieve only top K relevant chunks


from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    add_start_index=True
)

chunks = text_splitter.split_documents(pdfdocs)

In [37]:
print(f"length of chunks: {len(chunks)}")

# for chunk in chunks[:2]:
#     print(f"Chunk Content:\n")
#     print(f"{chunk.page_content}")
#     print(f"\nMetadata: {chunk.metadata}")
#     print("-"*100, "\n")

length of chunks: 19


In [38]:
# Create IDs for chunks, in order to keep only unique entries in vector store
# Chroma vector store does not handle duplicate entries

import hashlib

def generate_content_hash(chunk):
    chunk_content = chunk.page_content
    return hashlib.sha256(chunk_content.encode()).hexdigest()

chunk_ids = [generate_content_hash(chunk) for chunk in chunks]
print(f"Length of chunk ids: {len(chunk_ids)}")
print(f"Sample Chunk id: {chunk_ids[0]}")

Length of chunk ids: 19
Sample Chunk id: 8beefbb6a6bac8be09489f3504cc582d4fd368003236d272d324773906634990


In [39]:
# Embeddings [Convert words as vectors]
# This embedding object will be passed as a parameter to vector store constructor
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Simple embedding query
vector = embeddings.embed_query(chunks[0].page_content)

print(vector[0:5])

[-0.0083840685, 0.0008452388, 0.03825788, -0.060094304, -0.015525679]


In [ ]:
# Instanstiate vector store
#   - Store and retrieve documents

from langchain_chroma import Chroma

# Instanstiated vector store
vector_store = Chroma(
    collection_name="Langchain_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

"""
Who converts chunks into vectors?
- The embedding model (the embeddings object).

Who triggers the conversion?
- Chroma, when you call add_documents().

What happens internally:
- You pass text chunks to vector_store.add_documents()
- Chroma calls embeddings.embed_documents(chunks)
- The embedding model converts text → vectors
- Chroma stores those vectors in the vector database
"""

# delete_collection will delete all the chunks from vector store
# vector_store.delete_collection()

# All the chunks are added to vector store
lst_ids = vector_store.add_documents(documents=chunks, ids=chunk_ids)

# print(len(lst_ids))
# print(lst_ids[:3])

# Issue i faced: Multiple times the same chunks is added to vector store
#   - To avoid this, we can use the ids to check if the chunks are already added
#   - If the chunks are already added, then we can skip adding them again
#   - If the chunks are not added, then we can add them


19
['8beefbb6a6bac8be09489f3504cc582d4fd368003236d272d324773906634990', 'e1a301a00235a6cb5db6ff6bf645e73203dd0a9625ec0e5588cccc6f25a7f02e', '6e3114965aa04f170471681d1e9163d368c3d5f968a5adf7c8ec54427ae88b43']


In [51]:
# Different ways of searching in vector store
#   - similarity_search
#   - asimilarity_search
#   - similarity_search_with_score
#   - embed_query

# similarity_search will return list of documents
# List of questions
# "How many years of experience do kapil have?, give me only years of experience"
# "List all the skills of kapil"
# "List all the projects of kapil"
# "List all the companies kapil worked with"

results = vector_store.similarity_search(
    "List all the companies in the document"
)

print(len(results))
for result in results:
    print(result.page_content)
    print("-"*100, "\n")

4
ticket. 
• Interact with PLM team (SIEMENS and SAP) to analyze PLM and CAD integrated  
behavior and further analyze any workflows or requirements 
 
Company: CLOUDIX Global Solutions Private Limited, Chennai 
Date: April 2022 to Aug 2022 
Designation: DevOps Engineer 
 
Project: 
1. Vera (Health care) 
2. Mediguru (Health care) 
3. Palo-Alto (Data Analysis) 
 
 
 
 
2
---------------------------------------------------------------------------------------------------- 

to reduced DevOps team dependencies for household activities like getting logs, 
monitor status of apps in one dashboard. 
• Plan and setup environment for Palo-Alto team to help in development process 
and other DevOps activities. 
 
 
Company: P3 DESIGN SOLUTIONS PVT LTD, Chennai 
On Site: Mahindra Research Valley, Chennai 
Date: June 2017 to July 2019 
Designation: Software Engineer 
 
Project: 
1. NX addon for Configurable design platform (Mahindra and Swaraj)
------------------------------------------------------

In [53]:
# Get similarity search results asychronously
results = await vector_store.asimilarity_search("List all the companies name in the document")

print(len(results))
for result in results:
    print(result.page_content)
    print("-"*100, "\n")

4
to reduced DevOps team dependencies for household activities like getting logs, 
monitor status of apps in one dashboard. 
• Plan and setup environment for Palo-Alto team to help in development process 
and other DevOps activities. 
 
 
Company: P3 DESIGN SOLUTIONS PVT LTD, Chennai 
On Site: Mahindra Research Valley, Chennai 
Date: June 2017 to July 2019 
Designation: Software Engineer 
 
Project: 
1. NX addon for Configurable design platform (Mahindra and Swaraj)
---------------------------------------------------------------------------------------------------- 

ticket. 
• Interact with PLM team (SIEMENS and SAP) to analyze PLM and CAD integrated  
behavior and further analyze any workflows or requirements 
 
Company: CLOUDIX Global Solutions Private Limited, Chennai 
Date: April 2022 to Aug 2022 
Designation: DevOps Engineer 
 
Project: 
1. Vera (Health care) 
2. Mediguru (Health care) 
3. Palo-Alto (Data Analysis) 
 
 
 
 
2
------------------------------------------------------

In [54]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("List all the companies names")

print(f"length of result: {len(results)}\n\n")

for result in results:
    doc, score = result
    print(f"Score: {score}\n")
    print(doc.page_content)
    print("-"*100, "\n")

length of result: 4


Score: 0.7257918119430542

to reduced DevOps team dependencies for household activities like getting logs, 
monitor status of apps in one dashboard. 
• Plan and setup environment for Palo-Alto team to help in development process 
and other DevOps activities. 
 
 
Company: P3 DESIGN SOLUTIONS PVT LTD, Chennai 
On Site: Mahindra Research Valley, Chennai 
Date: June 2017 to July 2019 
Designation: Software Engineer 
 
Project: 
1. NX addon for Configurable design platform (Mahindra and Swaraj)
---------------------------------------------------------------------------------------------------- 

Score: 0.745211660861969

ticket. 
• Interact with PLM team (SIEMENS and SAP) to analyze PLM and CAD integrated  
behavior and further analyze any workflows or requirements 
 
Company: CLOUDIX Global Solutions Private Limited, Chennai 
Date: April 2022 to Aug 2022 
Designation: DevOps Engineer 
 
Project: 
1. Vera (Health care) 
2. Mediguru (Health care) 
3. Palo-Alto (Data An

In [ ]:
embedding = embeddings.embed_query("List all the companies kapil worked for")

results = vector_store.similarity_search_by_vector(embedding)

print(f"length of result: {len(results)}\n\n")

for result in results:
    print(result.page_content)
    print("-"*100, "\n")

In [55]:
# LangChain VectorStore objects do not subclass Runnable. 
# LangChain Retrievers are Runnables, so they implement a standard set of methods 
#       (e.g., synchronous and asynchronous invoke and batch operations). 
# Although we can construct retrievers from vector stores, retrievers can interface with 
#       non-vector store sources of data, as well (such as external APIs).

# "How many years of experience do kapil have?, give me only years of experience"
# "List all the skills of kapil"
# "List all the projects of kapil"
# "List all the companies kapil worked with"

from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)

lst_query =     [
        "Give me the total years of experience",
        "List his skills",
        "Give me the project information he worked",
        "List all the companie names"
    ]

results = retriever.batch(
    lst_query,
)

for index, result in enumerate(results, start=1):
    print(f"{index}. {lst_query[index-1]}\n")

    for doc in result:
        print(doc.page_content)
        print("-"*100, "\n")

    print("#"*100, "\n")


1. Give me the total years of experience

Skills 
Certification • Asp.Net developer 
• HashiCorp Certified: Terraform Associate 
Education Graduation: Bachelor of Engineering – BE, Mechanical Engineering 
University: Anna University 
Institution: SRG Engineering College 
Secured Marks: 7.7(CGPA) 
Year of Passing:  2013 
 
Language / Scripting C++, C# / Shell Scripting, Batch Scripting 
SDK / Framework CAD, Windows, MIP 
Cloud Management Tool AWS, Microsoft Azure 
SCM Tool Git, GitLab 
CI & Deployment Tool Jenkins, Azure DevOps
---------------------------------------------------------------------------------------------------- 

#################################################################################################### 

2. List his skills

Skills 
Certification • Asp.Net developer 
• HashiCorp Certified: Terraform Associate 
Education Graduation: Bachelor of Engineering – BE, Mechanical Engineering 
University: Anna University 
Institution: SRG Engineering College 
Secured Mar